In [4]:
# Classify an original DTI value into one of the six analysis groups.
def find_dti_group(dti):
   
    # Missing and exempt DTI values are excluded from grouped ststistics.
    if dti in ["","NA","Exempt"]:
        group=None
   
    # Keep the interval labels already provided in the original data.
    elif dti=="<20%":
        group="<20%"
    elif dti =="20%-<30%":
        group="20%-<30%"
    elif dti=="30%-<36%":
        group="30%-<36%"
    elif dti=="50%-60%":
        group="50%-60%"
    elif dti==">60%":
        group=">60%"
   
    else:
       # Convert individual DTI values from text to integers.
        dti_number=int(dti)
      
        # Combine values from 36 to 49 into the same analysis group.
        if dti_number>=36 and dti_number<50:
            group="36%-<50%"
        else:
            group=None
    return group

In [6]:
# Store results for one DTI group within a state and loan type.
class StateData:
    def __init__(self, state, loan_type, dti_bin, approval_rates, size):
        self.state = state  
        self.loan_type = loan_type 
        self.dti_bin = dti_bin 
        
        # Approval rate is a proportion between 0 and 1,
        # or None when the group has no valid applications.
        self.approval_rates = approval_rates
       
        # Number of valid applications used as the approval-rate denominator.
        self.size = size

    # Return a text summary of the stored result.
    def print_data(self):
        return f"state is {self.state}, Loan Type is {self.loan_type}, dti_bin is, {self.dti_bin}, Approval Rates is {self.approval_rates}, Size is {self.size}"

In [7]:
# Save each DTI group's result for comparison during the current session.
def data_saver(state, loan_type, dti_bins, approval_rates, size):

    # Store each group's label, approval rate and valid application count.
    for i in range(len(dti_bins)):
        new_data = StateData(state, loan_type, dti_bins[i], approval_rates[i], size[i])
        
        # Generate a dictionary key from the state, loan tyoe and DTI group.       
        new_data_hash = hash((state, loan_type, dti_bins[i]))
        
        # Ask before replacing an existing result.
        if (new_data_hash in saved_datasets_v2):
            overideYN = input(f"You are trying to save an existing record: {new_data.print_data()}, you sure want to overide? Y/N")

            # Replace the saved result only when the user enters Y or y.
            if (overideYN == "Y" or overideYN == "y"):
                saved_datasets_v2[new_data_hash] = new_data
                new_data.print_data()
        else:
            # Save the result directly if its key is not already present.
            saved_datasets_v2[new_data_hash] = new_data
            new_data.print_data()

In [8]:
# Select saved results for a given state and loan type.
# Return an array with columns: DTI group,approval rate, valid application count.
def list_object_to_array_and_filter(list_state_data, state, loan_type):
    array = []
    for state_data in list_state_data:

        # Keep only results matching both the state and loan type.
        if (state_data.state == state and state_data.loan_type == loan_type):
            array.append([state_data.dti_bin, state_data.approval_rates, state_data.size])
    
    # Allow text labels, numeric values and None in the same array.
    return np.array(array, dtype=object)

In [9]:
# Calculate valid application counts, approval counts and rates by DTI group.
# Each input record contains: DTI, loan type, action taken.
# Action codes are stored as strings.
def calculate_approval_rates(records):
    
    #keep a fixed group order for reporting, plotting and comparison.
    groups=["<20%",
           "20%-<30%",
           "30%-<36%",
           "36%-<50%",
           "50%-60%",
           ">60%"]

    # Start the valid and approved application counts at zero for each group.
    totals=[0,0,0,0,0,0]
    approved_counts=[0,0,0,0,0,0]

    for record in records:
        dti=record[0]
        action=record[2]
        group=find_dti_group(dti)

       # Only  action codes 1,2 and 3 enter the approval-rate denominator.
        if action in ["1","2","3"]:
            
            # Count the application only if its DTI matches an analysis group.
            for i in range(len(groups)):
                if group==groups[i]:
                    totals[i]+=1

                    # Action codes 1 and 2 count as approved applications.
                    if action in ["1","2"]:
                        approved_counts[i]+=1

    rates=[]

    # Calculate each rate as approved applications divided by valid applications. 
    for i in range(len(groups)):
        if totals[i] > 0:
            rate = approved_counts[i]/totals[i]
        else:
            # No valid applications means the rate is undefined, not zero.
            rate = None

        rates.append(rate)
    
    # Return four lists in the same DTI group order.
    # Rates are proportions between 0 and 1, or None for empty groups.
    return groups, totals, approved_counts, rates

In [11]:
import numpy as np

# Analyse and plot DTI approval rates for a selected state and loan type.
# Users can optionally save the results for later comparison.
def analyze_state_DTI_approval_rate():
    path = "2025_combined_mlar_header.txt"
    
    # Identify the dour fields required for the analysis.
    selected_columns = [
        "state_code",
        "loan_type",
        "action_taken",
        "debt_to_income_ratio"
    ]
    
    # Read the header to locate the required columns by name.
    with open(path, "r", encoding="utf-8-sig") as f:
        header = f.readline().strip().split("|")
    
    column_indices = [
        header.index(col) for col in selected_columns
    ]
    
    # Reas the first 9,998 records , keeping all selected values as text.
    # this preserves the original DTI interval labels.
    extracted_data = np.genfromtxt(
        path,
        delimiter="|",
        skip_header=1,
        usecols=column_indices,
        max_rows=9998,
        dtype=str,
        encoding="utf-8"
    )
    
    # Count missing values in each selected column for checking.
    missing_mask = np.isin(
        np.char.strip(extracted_data),
        ["", "NA", "nan", "NaN"]
    )
    
    for i, name in enumerate(selected_columns):
        print(name, "missing:", np.sum(missing_mask[:, i]))
    
    # Reorder columns to : DTI, state, loan tyoe, action taken.
    data = extracted_data.copy()
    data = data[:, [3, 0, 1, 2]]
    
    # Detect missing DTI and state values without changing the original data.
    dti_missing = np.isin(
        np.char.upper(np.char.strip(data[:, 0].astype(str))),
        ["", "NA", "NAN", "NONE", "NULL"]
    )
    
    state_missing = np.isin(
        np.char.upper(np.char.strip(data[:, 1].astype(str))),
        ["", "NA", "NAN", "NONE", "NULL"]
    )
    
    # Exclude rows with missing DTI or state information.
    # Exampt DTI values are handled later by find_dti_group().
    valid_rows = ~(state_missing | dti_missing)
    final_data = data[valid_rows].copy()
    
    print(final_data)
    
    sample_data = final_data
    
    # List the states available in the loaded data.
    city=sample_data[:,1]
    unique_city=np.unique(city)
    print("unique_city")
    # print(unique_city)
    city_list=unique_city.tolist()
    print(city_list)
    
    # Remove surrounding spaces and accpet lowercase state codes.
    state=input("Enter your state:").strip().upper()

    # Keep asking until the user selects an available state.
    while state not in city_list:
        print("Invalid choice. Please try again")
        state=input("Enter your state:").strip().upper()
    print("selected state")    
    print(state)
    
    # List the loan types available in the loaded data.
    loan=sample_data[:,2]
    unique_loan=np.unique(loan)
    print("unique loan")
    print(unique_loan)
    loan_list=unique_loan.tolist()
    loan_list= [str(x) for x in loan_list]
    # print(loan_list)

    # Keep the input as text to match the loan codes read from the file.
    loan_type=input("Enter your loan type:").strip()
    
    # keeping asking until the user selects an available loan type.
    while loan_type not in loan_list:
        print("Invalid choice. Please try again")
        loan_type=input("Enter your loan type:").strip()
    print("loan_type selected:")    
    print(int(loan_type))
    
    # create a Ture/False condition for the selected state.
    state_condition=city==state
    # print(state_condition)
    selected_state=sample_data[state_condition]
    # print(selected_state)
    # loan_type=int(loan_type)
    
    # Create a separate condition for the selected loan type.
    loan_type_condition=loan==loan_type
    # print(loan_type_condition)
    selected_loan_type=sample_data[loan_type_condition]
    # print(loan_type_condition)
    # print(selected_loan_type)
    
    # keep only rows matching both the statr and loan type.
    combined=state_condition&loan_type_condition
    filtered_data=sample_data[combined]
    # print(filtered_data)
    
    # the filtered data contains: DTI, state, loan type,action taken.
    records = filtered_data
    
    # Remove the state column before passing records to the calculation function.
    # The remaining columns are: DTI, loantype, action taken.
    records = np.delete(records, 1, axis=1)
    
    # Calculate counts and approval rates in the fixed DTI group order. 
    groups,totals,approved_counts,rates=calculate_approval_rates(records)
    
    # Convert valid application counts to a Numpy array for savinf and comparison. 
    size=np.array(totals)
    
    # Display the calculated results for checking.
    print("Groups:",groups)
    print("Totals:",totals)
    print("Approved:",approved_counts)
    print("Rates:",rates)
    print("Size:",size)
    
    # groups,totals,approved_counts,rates=calculate_approval_rates(check_records)
    
    # size=np.array(totals)
    
    # # print("Totals:",totals)
    # # print("Approved:",approved_counts)
    # # print("Rates:",rates)
    # # print("Size:",size)
    
    # empty_records=[]
    
    # groups,totals,approved_counts,rates=calculate_approval_rates(empty_records)
    
    # # print("Totals:",totals)
    # # print("Approved:",approved_counts)
    # # print("Rates:",rates)
    
    import matplotlib.pyplot as plt
    
   # Plot approval rates in the same DTI group order used in the calcultaion.
    plt.figure(figsize=(10, 6))
    
    
    plt.plot(groups, rates, marker='o', linewidth=2.5, markersize=8, color='#4C72B0')
    

    # Label groups with calculable rates, including their valid application counts.
    # Skip empty groups whose approval rate is None.
    for i in range(len(groups)):
        if rates[i] != None:
            label_text = f"{rates[i]*100:.1f}%\n(n={totals[i]})"

            # Place each label slightly above its plotted point.
            plt.text(i, rates[i] + 0.03, label_text, ha='center', va='bottom', fontsize=10, fontweight='bold')
    

    plt.title("DTI Bin vs Loan Approval Rate", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("DTI Bin", fontsize=12)
    plt.ylabel("Approval Rate", fontsize=12)
    
    # Display proportions as percentages on the vertical axis.
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    
    # Leave space above the points for their labels.
    plt.ylim(-0.05, 1.1) 
    plt.grid(True, linestyle='--', alpha=0.6)
    
   
    plt.tight_layout()
    plt.show()

    # Ask whether to store the results in memory for later comparison.
    save_data_YN = input("would you like to save the result for comparision?")
    if (save_data_YN == "Y" or save_data_YN == "y"):
        data_saver(state, loan_type, groups, rates, size)

In [12]:
# Compare approval outcomes for two saved states with the same loan type.
def comparison_analysis(state_1, state_2, loan_type):
    from scipy.stats import chi2_contingency
    state_1_array = []
    state_2_array = []
    
    # Retrieve each state's saved results for the selected loan type.
    # Columns are: Dti group, approval rate, valid application count.
    state_1_array = list_object_to_array_and_filter(saved_datasets_v2.values(), state_1, loan_type)
    state_2_array = list_object_to_array_and_filter(saved_datasets_v2.values(), state_2, loan_type)

    # Recover approved application counts from approval rates and group sizes.
    # Convert undefined products for empty groups to zero.
    state_1_passing_freq = np.nan_to_num(state_1_array[:,1].astype(float) * state_1_array[:,2].astype(float))

    # The remaining valid applicatons are non-approved applications.
    state_1_failed_freq = state_1_array[:,2].astype(float) - state_1_passing_freq

    # Build a table with DTI groups as rows.
    # The two columns contain approved and non-approved application counts.
    chi_table_1 = np.column_stack((state_1_passing_freq, state_1_failed_freq))

    # Test the association between DTI group and approval outcome in state 1.
    try:
        chi2_1, p_1, dof_1, expected_1 = chi2_contingency(chi_table_1)
    except ValueError:
        print("Insufficient data!")

    # Prepare the same application counts for state 2.
    state_2_passing_freq = np.nan_to_num(state_2_array[:,1].astype(float) * state_2_array[:,2].astype(float))
    state_2_failed_freq = state_2_array[:,2].astype(float) - state_2_passing_freq

    chi_table_2 = np.column_stack((state_2_passing_freq, state_2_failed_freq))
    
    # Test the association between DTI group and approval outcome in state 2.
    try:
        chi2_2, p_2, dof_2, expected_2 = chi2_contingency(chi_table_2)
    except ValueError:
        print("Insufficient data!")
        
    print("chi2_2, p_2, dof_2, expected_2:", chi2_2, p_2, dof_2, expected_2)


    # Combine all DTI groups to obtain each state's overall application counts.
    state_1_passing_sum = state_1_passing_freq.sum()
    state_1_failed_sum = state_1_failed_freq.sum()
    state_2_passing_sum = state_2_passing_freq.sum()
    state_2_failed_sum = state_2_failed_freq.sum()

    # Build a 2-by-2 table.
    # Rows represent the two states; columns represent approved
    # and non-approved application counts.
    homogeneity_table = np.array([[ state_1_passing_sum, state_1_failed_sum],[state_2_passing_sum, state_2_failed_sum]])
    
    # Test whether overall approval proportions differ between the two states.
    # Yates' continuity correction is disabled.
    chi_h, p_h, dof_h, expected_h = chi2_contingency(homogeneity_table, correction = False)
    
    # Display the test statistic, p-value, degrees of freedom and expected counts.
    print("chi_h, p_h, dof_h, expected_h", chi_h, p_h, dof_h, expected_h)


    # Compare the two states separately within each matching DTI group.
    same_bin_test_list = []
    
    # Both states' results must follow the same DTI group order.
    for i in range(len(state_1_passing_freq)):
        
        # Build a 2-by-2 tables for current DTI group.
        # Rows represent states; columns represent approved.
        # and non-approved application counts.
        same_bin_test_table = np.array([
            [state_1_passing_freq[i],state_1_failed_freq[i]], [state_2_passing_freq[i], state_2_failed_freq[i]]
        ])
        
        # Test whether approval proportions differ between states in this group.
        chi2_bin, p_bin, dof_bin, expected_bin = chi2_contingency(same_bin_test_table)
        
        # Store the test statistics, p-value, degrees of freedom and expected counts.
        same_bin_result = [chi2_bin, p_bin, dof_bin, expected_bin]
        same_bin_test_list.append(same_bin_result)

    # Dispaly test results in the original DTI group order.
    print("same_bin_test_list:", same_bin_test_list)    

In [13]:

# Read user commands and run the requested function until the user quits.
def cli():
    while True:
        command = input("please input your command")

        # Analyse a single state and optionally save its resultes.
        if (command == "/analyzeSingleState"):
            analyze_state_DTI_approval_rate()
            continue
            
        # Compare two states whose results have already been saved.
        elif (command == "/comparisionAnalysis"):
            state_1 = input("input the first state you would like to analyze, state need to be previously saved")
            state_2 = input("input the second state you would like to analyze, state need to be previously saved")
            loan_type = input("input the loan type you would like to analyze")
            comparison_analysis(state_1,state_2,loan_type) 
            continue

        # Leave the sommand loop.
        elif (command == "/q"):
            break

        # Clear results stores in memory during the current session.
        elif (command == "/flush"):
            saved_datasets_v2.clear()
            continue

        # Display the available sommands and their purpoes.
        elif (command == "/help"):
            print("available commandsL")
            print("/analyzeSingleState","to analyze a single state, you will have the opportunity to save the result at the end of it")
            print("/comparisionAnalysis", "to analyze two states from saved data")
            print("/flush", "to clear the saved results")
            print("/q", "to quit")
        
        # Prompt again of the sommand is not recoginsed.
        else:
            print("Instruction unclear")
            continue

# Created the result storage and start the command interface.
def main():
    
    # make the same result dictionary acceddible to the saving and comparison functions.
    global saved_datasets_v2
    
    # Start with empty storgae each time main() is called.
    saved_datasets_v2 = {}
    cli()

In [ ]:
# start the interactive programme.
main()

In [ ]:
# column_names = [
#     "state",
#     "loan_type",
#     "action_taken",
#     "debt_to_income"
# ]

# import numpy as np

# data = extracted_data.copy()

# data = data[:, [3, 0, 1, 2]]

# dti_missing = np.isin(
#     np.char.upper(np.char.strip(data[:, 0].astype(str))),
#     ["", "NA", "NAN", "NONE", "NULL"]
# )

# state_missing = np.isin(
#     np.char.upper(np.char.strip(data[:, 1].astype(str))),
#     ["", "NA", "NAN", "NONE", "NULL"]
# )

# valid_rows = ~(dti_missing | state_missing)

# final_data = data[valid_rows].copy()

# print("Original shape:", data.shape)
# print("DTI missing:", np.sum(dti_missing))
# print("State missing:", np.sum(state_missing))
# print("Deleted rows:", np.sum(~valid_rows))
# print("Final shape:", final_data.shape)

# print("Remaining DTI missing:", np.sum(dti_missing[valid_rows]))
# print("Remaining State missing:", np.sum(state_missing[valid_rows]))

# print(final_data[:10])

# import numpy as np

# column_names = ["DTI", "State", "LoanType", "action_taken"]

# for i, name in enumerate(column_names):
#     values = np.char.upper(
#         np.char.strip(final_data[:, i].astype(str))
#     )

#     missing = np.isin(
#         values,
#         ["", "NA", "NAN", "NONE", "NULL"]
#     )

#     print(name, "missing values:", np.sum(missing))

# print("Final data shape:", final_data.shape)

# import numpy as np

# reordered_data = cleaned_data[:, [3, 0, 1, 2]]

# print(reordered_data[:10])
# print(reordered_data.shape)
# print(type(reordered_data))

# import numpy as np

# column_names = ["DTI", "State", "LoanType", "action_taken"]

# for i, name in enumerate(column_names):
#     values = np.char.upper(
#         np.char.strip(final_data[:, i].astype(str))
#     )

#     missing = np.isin(
#         values,
#         ["", "NA", "NAN", "NONE", "NULL"]
#     )

#     print(name, "missing values:", np.sum(missing))

# print("Final data shape:", final_data.shape)

# total_records = extracted_data.shape[0]
# retained_records = final_data.shape[0]
# excluded_records = total_records - retained_records

# print("Total records read:", total_records)
# print("Excluded records:", excluded_records)
# print("Retained records:", retained_records)

# print("Exclusion rate:", round(excluded_records / total_records * 100, 2), "%")
# print("Retention rate:", round(retained_records / total_records * 100, 2), "%")

# print(data.shape)
# print(data.dtype.names)